# Notebook 11 - Structure-distance fixture builder

Builds the reusable perturbation fixture that the E07 structure-distance experiments
(`notebooks/experiments/E07-kj-structure-distance.ipynb`) and any later structure work read from.
The fixture is content-and-structure controlled: byte-identical reorders isolate the order axis,
the natural cross-summary pairs supply the realistic diffuse-`T` regime, and paragraph / heading
blocks supply the section axis. No embeddings are baked in - those are recomputed downstream under
the raw single-pair regime the design gates on.

**Approach**

1. Segment the 11 IBM exec-summaries and the source article into statements per block (paragraph for
   summaries, heading-section for the source) so every statement carries an order index and a block label
2. Build the byte-identical reorder pool - seeded k-swap permutations sweeping displacement from identity
   to near-reversal, each tagged with its normalized Spearman-footrule displacement
3. Build the natural cross-summary pair list (real no-exact-twin paraphrases) and the tier-contrast pairs
   (gold / adv1 / adv2) for the content guardrail
4. Build the section-swap perturbation (relocate statements across block boundaries) for the section axis
5. Save statements, reorder pool, pairs and metadata as JSON under `data/processed/structure-fixture/`

**Output** - `data/processed/structure-fixture/{statements,reorder_pool,pairs,meta}.json`

## GPU selection

This is a data-prep notebook - the only model is the `sat-3l-sm` statement segmenter (wtpsplit, CPU);
no embeddings are computed here. The GPU is pinned by UUID only so any lazy torch import lands on the
intended card, matching the E07 experiment notebook.

In [1]:
import os

# RTX 5000 Ada (GPU 2) by UUID - segmentation is CPU-light, this only pins a lazy torch import
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-c15a4c9a-8c2c-7fb9-a46b-fe4dff5dacf4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import warnings
warnings.filterwarnings("ignore")
print("fixture builder - segmentation only (CPU); GPU pinned to RTX 5000 Ada by UUID for consistency")

fixture builder - segmentation only (CPU); GPU pinned to RTX 5000 Ada by UUID for consistency


## Imports

In [2]:
%load_ext autoreload
%autoreload 2

import re, json, itertools
from pathlib import Path

import numpy as np
from rich.console import Console
from rich.table import Table

from docdistance.encoders import Segmenter

console = Console()
print("imports ready")

imports ready


## Reproducibility

In [3]:
SEED = 0
np.random.seed(SEED)
print(f"SEED={SEED}")

SEED=0


## Configuration

Paths, the 11-document manifest (same as E06), the reorder-sweep grid, and the section-perturbation
grid in one place. The reorder sweep generates a pool that E07 bins into 6 equal-width displacement
bins; the grid is sized so every bin holds >= 10 permutations after binning.

In [4]:
ROOT = Path("/home/lab/workspace/learning/projects/docdistance")
SUMMARY_DIR = ROOT / "data/interim/exec-summaries/ibm-ai-adoption/summaries"
SOURCE_FILE = ROOT / "data/interim/exec-summaries/ibm-ai-adoption/source/source-article.md"
OUT_DIR = ROOT / "data/processed/structure-fixture"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# same fixture as E06 - (filename, label, tier)
DOCS = [
    ("exec-summary-gold-opus-4-8.md",     "gold",   "gold"),
    ("exec-summary-gold-2-opus-4-8.md",   "gold-2", "gold"),
    ("exec-summary-1-opus-4-8.md",        "v1",     "gold"),
    ("exec-summary-2-opus-4-8.md",        "v2",     "gold"),
    ("exec-summary-opus-4-8.md",          "opus",   "gold"),
    ("exec-summary-sonnet-4-6.md",        "sonnet", "gold"),
    ("exec-summary-haiku-4-5.md",         "haiku",  "gold"),
    ("exec-summary-adv1-a-sonnet-4-6.md", "adv1-a", "adv1"),
    ("exec-summary-adv1-b-sonnet-4-6.md", "adv1-b", "adv1"),
    ("exec-summary-adv2-a-haiku-4-5.md",  "adv2-a", "adv2"),
    ("exec-summary-adv2-b-haiku-4-5.md",  "adv2-b", "adv2"),
]
ANCHOR = "gold"

# reorder sweep - gold-tier summaries are the permutation bases (clean order isolation)
BASE_DOCS = ["gold", "gold-2", "v1", "v2", "opus", "sonnet", "haiku"]
SWAP_KS = [0, 1, 2, 3, 4, 6, 8, 11, 15, 20, 28, 40]   # number of random transpositions
SEEDS_PER_K = 14
N_BINS = 6                                            # E07 bins displacement into this many

# section axis - relocate k statements across block boundaries
SECTION_K = [1, 2, 3]
SECTION_SEEDS = 10

t = Table(title="Notebook 11 configuration", title_style="bold cyan", show_header=False, box=None, padding=(0, 2))
t.add_column(style="bold cyan"); t.add_column()
t.add_row("Documents", f"{len(DOCS)} summaries + 1 source")
t.add_row("Reorder bases", f"{len(BASE_DOCS)} gold-tier ({', '.join(BASE_DOCS)})")
t.add_row("Swap grid", f"{SWAP_KS} x {SEEDS_PER_K} seeds + reversal anchors")
t.add_row("Displacement bins", f"{N_BINS} equal-width on normalized footrule (binned in E07)")
t.add_row("Section perturbation", f"relocate k in {SECTION_K}, {SECTION_SEEDS} seeds")
t.add_row("Output", str(OUT_DIR))
console.print(t)

                                          Notebook 11 configuration                                           
  Documents               11 summaries + 1 source                                                             
  Reorder bases           7 gold-tier (gold, gold-2, v1, v2, opus, sonnet, haiku)                             
  Swap grid               [0, 1, 2, 3, 4, 6, 8, 11, 15, 20, 28, 40] x 14 seeds + reversal anchors             
  Displacement bins       6 equal-width on normalized footrule (binned in E07)                                
  Section perturbation    relocate k in [1, 2, 3], 10 seeds                                                   
  Output                  /home/lab/workspace/learning/projects/docdistance/data/processed/structure-fixture

## Data loading and segmentation

Each document is split into blocks first (paragraph for summaries, heading-section for the source),
then each block is segmented into statements, so every statement carries both an order index (its
position in the flattened sequence) and a block label. The summaries carry one markdown heading each,
so paragraphs are the block unit for them (pre-registered); the source article carries 13 headings.

In [5]:
def split_blocks(text, mode):
    """Return [(block_name, block_text)] - paragraphs (blank-line) or heading-sections."""
    if mode == "para":
        parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
        return [(f"p{idx}", p) for idx, p in enumerate(parts)]
    # heading mode - a new block starts at each markdown heading line
    blocks, cur_name, cur = [], "preamble", []
    for line in text.splitlines():
        if re.match(r"^#{1,6}\s+", line):
            if any(c.strip() for c in cur):
                blocks.append((cur_name, "\n".join(cur).strip()))
            cur_name = line.lstrip("# ").strip()[:48]
            cur = [line]
        else:
            cur.append(line)
    if any(c.strip() for c in cur):
        blocks.append((cur_name, "\n".join(cur).strip()))
    return blocks

seg = Segmenter()

def build_doc(text, mode):
    statements, block_id, block_name = [], [], []
    for bidx, (bname, btext) in enumerate(split_blocks(text, mode)):
        for s in seg.split(btext):
            statements.append(s); block_id.append(bidx); block_name.append(bname)
    return statements, block_id, block_name

DOCSTORE = {}
for filename, label, tier in DOCS:
    st, bid, bnm = build_doc((SUMMARY_DIR / filename).read_text(), "para")
    DOCSTORE[label] = {"statements": st, "block_id": bid, "block_name": bnm,
                       "tier": tier, "kind": "summary", "n": len(st), "n_blocks": len(set(bid))}
st, bid, bnm = build_doc(SOURCE_FILE.read_text(), "heading")
DOCSTORE["source"] = {"statements": st, "block_id": bid, "block_name": bnm,
                      "tier": "source", "kind": "source", "n": len(st), "n_blocks": len(set(bid))}

t = Table(title="Segmented documents", title_style="bold cyan")
for c in ("label", "tier", "kind", "statements", "blocks"):
    t.add_column(c, style="cyan" if c == "label" else None)
for label, d in DOCSTORE.items():
    style = "bold green" if label == ANCHOR else None
    t.add_row(label, d["tier"], d["kind"], str(d["n"]), str(d["n_blocks"]), style=style)
console.print(t)
print(f"summaries blocks: min={min(DOCSTORE[l]['n_blocks'] for l in DOCSTORE if DOCSTORE[l]['kind']=='summary')}"
      f" (>=2 needed for the section axis); source blocks={DOCSTORE['source']['n_blocks']}")

                Segmented documents                
┏━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┓
┃ label  ┃ tier   ┃ kind    ┃ statements ┃ blocks ┃
┡━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━┩
│ gold   │ gold   │ summary │ 14         │ 6      │
│ gold-2 │ gold   │ summary │ 14         │ 6      │
│ v1     │ gold   │ summary │ 13         │ 4      │
│ v2     │ gold   │ summary │ 12         │ 4      │
│ opus   │ gold   │ summary │ 14         │ 6      │
│ sonnet │ gold   │ summary │ 13         │ 6      │
│ haiku  │ gold   │ summary │ 15         │ 5      │
│ adv1-a │ adv1   │ summary │ 14         │ 5      │
│ adv1-b │ adv1   │ summary │ 9          │ 4      │
│ adv2-a │ adv2   │ summary │ 14         │ 5      │
│ adv2-b │ adv2   │ summary │ 12         │ 6      │
│ source │ source │ source  │ 62         │ 13     │
└────────┴────────┴─────────┴────────────┴────────┘

summaries blocks: min=4 (>=2 needed for the section axis); source blocks=13


## Byte-identical reorder pool

The clean order-isolation fixture: a base document permuted into itself, so the transport plan `T` is
trivially sharp and the structure signal is pure order. Displacement is the normalized Spearman footrule
of the permutation; `k` random transpositions sweep it from 0 (identity) upward, reversal anchors fill
the top of the range. The pool is over-generated; E07 bins it into 6 equal-width bins and samples >= 10
per bin. This is the *upper bound* the design flags - the diffuse-`T` realism comes from the natural
cross-summary pairs below.

In [6]:
def norm_footrule(perm):
    """Normalized Spearman footrule of a permutation - 0 identity, ~1 reversal."""
    n = len(perm)
    if n < 2:
        return 0.0
    return float(np.abs(np.asarray(perm) - np.arange(n)).sum()) / (n * n // 2)

def k_swap(n, k, rng):
    p = np.arange(n)
    for _ in range(k):
        i, j = rng.integers(0, n), rng.integers(0, n)
        p[i], p[j] = p[j], p[i]
    return p

REORDER_POOL = {}
for bi, label in enumerate(BASE_DOCS):
    n = DOCSTORE[label]["n"]
    pool = []
    for ki, k in enumerate(SWAP_KS):
        for s in range(SEEDS_PER_K):
            rng = np.random.default_rng(SEED * 1_000_000 + bi * 10_000 + ki * 100 + s)
            p = k_swap(n, k, rng)
            pool.append({"k": int(k), "seed": int(s), "perm": p.tolist(), "disp": norm_footrule(p)})
    # reversal anchors - reverse then 0-2 swaps, to populate the high-displacement bin
    rev = np.arange(n)[::-1].copy()
    for s in range(SEEDS_PER_K):
        rng = np.random.default_rng(SEED * 1_000_000 + bi * 10_000 + 99 * 100 + s)
        p = rev.copy()
        for _ in range(int(rng.integers(0, 3))):
            i, j = rng.integers(0, n), rng.integers(0, n)
            p[i], p[j] = p[j], p[i]
        pool.append({"k": -1, "seed": int(s), "perm": p.tolist(), "disp": norm_footrule(p)})
    REORDER_POOL[label] = pool

# bin-occupancy check - confirm every one of the 6 bins holds >= 10 perms after binning
edges = np.linspace(0.0, 1.0, N_BINS + 1)
t = Table(title=f"Reorder pool - occupancy across {N_BINS} displacement bins", title_style="bold cyan")
t.add_column("base", style="cyan"); t.add_column("n", justify="right"); t.add_column("pool", justify="right")
for b in range(N_BINS):
    t.add_column(f"[{edges[b]:.2f},{edges[b+1]:.2f}]", justify="right")
ok = True
for label in BASE_DOCS:
    disps = np.array([e["disp"] for e in REORDER_POOL[label]])
    counts = [int(((disps >= edges[b]) & (disps < edges[b + 1] if b < N_BINS - 1 else disps <= edges[b + 1] + 1e-9)).sum())
              for b in range(N_BINS)]
    ok = ok and all(c >= 10 for c in counts)
    t.add_row(label, str(DOCSTORE[label]["n"]), str(len(disps)), *[str(c) for c in counts])
console.print(t)
print("every bin >= 10 perms:", ok, "| if a high bin is short, reversal anchors + small n cap it (noted in E07)")

                           Reorder pool - occupancy across 6 displacement bins                            
┏━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ base   ┃  n ┃ pool ┃ [0.00,0.17] ┃ [0.17,0.33] ┃ [0.33,0.50] ┃ [0.50,0.67] ┃ [0.67,0.83] ┃ [0.83,1.00] ┃
┡━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ gold   │ 14 │  182 │          40 │          28 │          32 │          40 │          25 │          17 │
│ gold-2 │ 14 │  182 │          42 │          33 │          28 │          38 │          24 │          17 │
│ v1     │ 13 │  182 │          33 │          31 │          29 │          44 │          26 │          19 │
│ v2     │ 12 │  182 │          35 │          25 │          25 │          48 │          34 │          15 │
│ opus   │ 14 │  182 │          44 │          20 │          36 │          36 │          29 │          17 │
│ sonnet │ 13 │  182 │          32 │          28 │          39 │          39 │          25 │          19 │
│ haiku  │ 15 │  182 │          44 │          21 │          31 │          42 │          31 │          13 │
└────────┴────┴──────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘

every bin >= 10 perms: True | if a high bin is short, reversal anchors + small n cap it (noted in E07)


## Natural pairs, tier contrasts, and section perturbation

Three more fixture pieces. The cross-summary pairs are real no-exact-twin paraphrases (11 independent
summaries of one article) - the realistic diffuse-`T` regime for the `T`-sparsity gate. The tier
contrasts give the content guardrail (gold vs gold = same content, gold vs adv1/adv2 = different content).
The section-swap relocates statements across block boundaries for the bounded section metric.

In [7]:
summary_labels = [l for l, d in DOCSTORE.items() if d["kind"] == "summary"]

# all natural cross-summary pairs (realistic diffuse-T)
cross_summary = [[a, b] for a, b in itertools.combinations(summary_labels, 2)]

# tier contrasts vs the gold anchor - same content (gold tier) vs different content (adv tiers)
tier_contrast = []
for b in summary_labels:
    if b == ANCHOR:
        continue
    tier_contrast.append({"a": ANCHOR, "b": b, "tier_b": DOCSTORE[b]["tier"],
                          "content": "same" if DOCSTORE[b]["tier"] == "gold" else "diff"})

# section-swap - relocate k statements to a different block; byte-identical statements, only block label moves
def section_swap(block_id, k, rng):
    n = len(block_id)
    blocks = sorted(set(block_id))
    if len(blocks) < 2:
        return None
    idx = rng.choice(n, size=min(k, n), replace=False)
    new_block = list(block_id)
    for i in idx:
        others = [b for b in blocks if b != block_id[i]]
        new_block[int(i)] = int(rng.choice(others))
    return {"moved_idx": [int(x) for x in idx], "new_block_id": [int(x) for x in new_block]}

section_swaps = {}
for bi, label in enumerate(BASE_DOCS):
    d = DOCSTORE[label]
    if d["n_blocks"] < 2:
        continue
    entries = []
    for ki, k in enumerate(SECTION_K):
        for s in range(SECTION_SEEDS):
            rng = np.random.default_rng(SEED * 2_000_000 + bi * 10_000 + ki * 100 + s)
            sw = section_swap(d["block_id"], k, rng)
            if sw:
                sw.update({"k": int(k), "seed": int(s)})
                entries.append(sw)
    section_swaps[label] = entries

PAIRS = {
    "cross_summary": cross_summary,
    "tier_contrast": tier_contrast,
    "section_summary_source": ["gold", "source"],
    "section_swap": section_swaps,
}
print(f"cross-summary pairs: {len(cross_summary)} | tier contrasts: {len(tier_contrast)}"
      f" (same={sum(t['content']=='same' for t in tier_contrast)}, diff={sum(t['content']=='diff' for t in tier_contrast)})")
print(f"section-swap bases: {list(section_swaps)} | summary<->source section pair: gold vs source")

cross-summary pairs: 55 | tier contrasts: 10 (same=6, diff=4)
section-swap bases: ['gold', 'gold-2', 'v1', 'v2', 'opus', 'sonnet', 'haiku'] | summary<->source section pair: gold vs source


## Save fixture

In [8]:
meta = {
    "seed": SEED, "n_bins": N_BINS, "swap_ks": SWAP_KS, "seeds_per_k": SEEDS_PER_K,
    "base_docs": BASE_DOCS, "section_k": SECTION_K, "section_seeds": SECTION_SEEDS,
    "anchor": ANCHOR, "block_unit": {"summary": "paragraph", "source": "heading"},
    "doc_counts": {l: {"n": d["n"], "n_blocks": d["n_blocks"], "tier": d["tier"], "kind": d["kind"]}
                   for l, d in DOCSTORE.items()},
    "note": "raw single-pair embeddings recomputed downstream (no anisotropy); reorder pool is the "
            "byte-identical upper bound, cross_summary supplies the realistic diffuse-T regime",
}
(OUT_DIR / "statements.json").write_text(json.dumps(DOCSTORE, ensure_ascii=False, indent=1))
(OUT_DIR / "reorder_pool.json").write_text(json.dumps(REORDER_POOL, indent=1))
(OUT_DIR / "pairs.json").write_text(json.dumps(PAIRS, ensure_ascii=False, indent=1))
(OUT_DIR / "meta.json").write_text(json.dumps(meta, indent=2))

t = Table(title="Fixture written", title_style="bold green", show_header=True)
t.add_column("file", style="cyan"); t.add_column("bytes", justify="right"); t.add_column("contains")
t.add_row("statements.json", str((OUT_DIR / "statements.json").stat().st_size), f"{len(DOCSTORE)} docs, statements + block labels")
t.add_row("reorder_pool.json", str((OUT_DIR / "reorder_pool.json").stat().st_size), f"{sum(len(v) for v in REORDER_POOL.values())} permutations over {len(BASE_DOCS)} bases")
t.add_row("pairs.json", str((OUT_DIR / "pairs.json").stat().st_size), f"{len(cross_summary)} cross-summary + {len(tier_contrast)} tier + section")
t.add_row("meta.json", str((OUT_DIR / "meta.json").stat().st_size), "seed, bins, grids, block units")
console.print(t)

                           Fixture written                           
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ file              ┃  bytes ┃ contains                             ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ statements.json   │  41185 │ 12 docs, statements + block labels   │
│ reorder_pool.json │ 228039 │ 1274 permutations over 7 bases       │
│ pairs.json        │  47984 │ 55 cross-summary + 10 tier + section │
│ meta.json         │   1876 │ seed, bins, grids, block units       │
└───────────────────┴────────┴──────────────────────────────────────┘

## Summary

The fixture is complete and reusable: segmented statements with block labels, the byte-identical reorder
pool sweeping displacement, the realistic cross-summary pairs, the tier content contrasts, and the
section-swap perturbation. E07 reads this without re-segmenting and recomputes embeddings under the raw
single-pair regime. Nothing here depends on an LLM - the deferred order-preserving paraphrase and
rewrite-only content-edit cells (no API key) are the only fixture pieces left for a follow-up.

In [9]:
assert all((OUT_DIR / f).exists() for f in ["statements.json", "reorder_pool.json", "pairs.json", "meta.json"])
print("fixture ready at", OUT_DIR)

fixture ready at /home/lab/workspace/learning/projects/docdistance/data/processed/structure-fixture
